# 🎯 OneVoice Edge — ASR Benchmark & Fine-Tuning
**Models:** SenseVoiceSmall (`iic/SenseVoiceSmall`) & GIPFormer (`g-group-ai-lab/gipformer-65M-rnnt`)
**Features:** Full Checkpoint Auto-Resume on Drive · Industrial Noise Adaptation · WER/CTER Metrics

## Cell 1 — Mount Drive & Install Project Dependencies

In [1]:
import os, sys
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
    MODEL_OUTPUT = '/content/drive/MyDrive/onevoice_models/sensevoice_finetuned'
else:
    DATASET_ROOT = './data/onevoice_audio_v1'
    MODEL_OUTPUT = './models/sensevoice_finetuned'

os.makedirs(MODEL_OUTPUT, exist_ok=True)
print(f'Dataset path: {DATASET_ROOT}')
print(f'Model save path: {MODEL_OUTPUT}')

# Install project ASR requirements cleanly
!pip install -q funasr modelscope funasr_onnx sherpa-onnx jiwer torchaudio soundfile librosa tqdm

# Verify C-extensions; if pip updated libraries in live process, auto-restart Python session
try:
    import pandas as pd, numpy as np
    pd.DataFrame({'a': [1]})
    print('✅ Python C-extensions verified OK!')
except Exception:
    print('🔄 Auto-restarting Python runtime to reload C-extensions...')
    os._exit(0)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset path: /content/drive/MyDrive/onevoice_audio_v1
Model save path: /content/drive/MyDrive/onevoice_models/sensevoice_finetuned


## Cell 2 — Clone Project Repository

In [2]:
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned. Pulling latest...')
    !cd /content/OneVoice && git pull

import sys
sys.path.append('/content/OneVoice/onevoice-edge')
print('✅ Project path linked.')

Repo already cloned. Pulling latest...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 6), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.17 KiB | 399.00 KiB/s, done.
From https://github.com/Platypus27-coder/OneVoice
   b50c545..ad831a1  main       -> origin/main
Updating b50c545..ad831a1
Fast-forward
 notebooks/colab_benchmark_and_finetune_asr.ipynb | 31 +++++++++++++++++-------
 1 file changed, 22 insertions(+), 9 deletions(-)
✅ Project path linked.


## Cell 3 — Load Manifest & Prepare Evaluation Sets

In [3]:
import os, json, pandas as pd

MANIFEST_PATH = os.path.join(DATASET_ROOT, 'manifest.jsonl')
CLEAN_DIR = os.path.join(DATASET_ROOT, 'clean')
NOISY_DIR = os.path.join(DATASET_ROOT, 'noisy')

assert os.path.exists(MANIFEST_PATH), f'Manifest not found at {MANIFEST_PATH}! Please check Cell 1.'

entries = []
with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            entries.append(json.loads(line.strip()))

df_manifest = pd.DataFrame(entries)
print(f'Total manifest samples loaded: {len(df_manifest)}')
print(df_manifest[['audio', 'text', 'noise_type', 'snr_db', 'split']].head())


Total manifest samples loaded: 16128
                audio                                    text noise_type  \
0  OV2_000001_n01.wav  Kiểm tra mũ bảo hộ trước khi tiếp tục.      truck   
1  OV2_000001_n02.wav  Kiểm tra mũ bảo hộ trước khi tiếp tục.   drilling   
2  OV2_000002_n01.wav        Mũ bảo hộ đã được kiểm tra chưa?       wind   
3  OV2_000002_n02.wav        Mũ bảo hộ đã được kiểm tra chưa?     hammer   
4  OV2_000003_n01.wav                Có vấn đề với mũ bảo hộ.       wind   

   snr_db  split  
0      15   test  
1      20   test  
2      15  train  
3      10  train  
4       5  train  


## Cell 4 — Benchmark Baseline Model (SenseVoice Small / GIPFormer)
Evaluates **WER** (Word Error Rate) and **CTER** (Construction Term Error Rate) on Clean vs Noisy audio.

In [ ]:
import re, glob, torch, jiwer
from tqdm.notebook import tqdm
from funasr import AutoModel

EVAL_CHECKPOINT_FILE = os.path.join(MODEL_OUTPUT, 'eval_baseline_results.jsonl')

print('⚙️ Loading Baseline SenseVoiceSmall model (iic/SenseVoiceSmall)...')
model = AutoModel(
    model='iic/SenseVoiceSmall',
    vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    vad_kwargs={'max_single_segment_time': 30000},
    device='cuda' if torch.cuda.is_available() else 'cpu',
    disable_update=True
)
print('✅ Model loaded successfully.')

def clean_transcript(text):
    text = re.sub(r'<\|.*?\|>', '', str(text)).lower()
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def evaluate_asr(sample_limit=200):
    # Check existing eval checkpoint
    evaluated_ids = set()
    eval_records = []
    if os.path.exists(EVAL_CHECKPOINT_FILE):
        with open(EVAL_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line.strip())
                    evaluated_ids.add(data['audio'])
                    eval_records.append(data)
        print(f'🔄 Resuming Eval Checkpoint: {len(evaluated_ids)} samples already evaluated.')

    test_df = df_manifest[df_manifest['split'] == 'test'] if 'split' in df_manifest else df_manifest
    if sample_limit:
        test_df = test_df.head(sample_limit)

    with open(EVAL_CHECKPOINT_FILE, 'a', encoding='utf-8') as ef:
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluating'):
            if row['audio'] in evaluated_ids:
                continue

            ref_text = clean_transcript(row['text'])
            if not ref_text: continue

            clean_path = os.path.join(CLEAN_DIR, row['clean_audio'])
            noisy_path = os.path.join(NOISY_DIR, row['audio'])

            pred_clean, pred_noisy = '', ''
            if os.path.exists(clean_path):
                res_c = model.generate(input=clean_path, cache={}, language='auto', use_itn=True)
                pred_clean = clean_transcript(res_c[0]['text'])

            if os.path.exists(noisy_path):
                res_n = model.generate(input=noisy_path, cache={}, language='auto', use_itn=True)
                pred_noisy = clean_transcript(res_n[0]['text'])

            rec = {'audio': row['audio'], 'ref': ref_text, 'pred_clean': pred_clean, 'pred_noisy': pred_noisy}
            ef.write(json.dumps(rec, ensure_ascii=False) + '\n')
            ef.flush()
            eval_records.append(rec)
            evaluated_ids.add(row['audio'])

    c_refs = [r['ref'] for r in eval_records if r['pred_clean']]
    c_preds = [r['pred_clean'] for r in eval_records if r['pred_clean']]
    n_refs = [r['ref'] for r in eval_records if r['pred_noisy']]
    n_preds = [r['pred_noisy'] for r in eval_records if r['pred_noisy']]

    wer_clean = jiwer.wer(c_refs, c_preds) * 100 if c_refs else 0.0
    wer_noisy = jiwer.wer(n_refs, n_preds) * 100 if n_refs else 0.0

    print('\n' + '='*50)
    print('📊 BASELINE ASR EVALUATION RESULTS:')
    print(f'  • WER (Clean Audio) : {wer_clean:.2f}%')
    print(f'  • WER (Noisy Audio) : {wer_noisy:.2f}%')
    print(f'  • Degradation Gap   : +{wer_noisy - wer_clean:.2f}% WER increase due to noise')
    print('='*50)

evaluate_asr(sample_limit=200)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


⚙️ Loading Baseline SenseVoiceSmall model (iic/SenseVoiceSmall)...
funasr version: 1.4.2.


2026-08-19 08:34:55,198 | INFO    | modelscope_hub.download | Downloading 20 files from iic/SenseVoiceSmall@master


Downloading:   0%|          | 0/20 [00:00<?, ?file/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

...

ser_figure.png:   0%|          | 0.00/199k [00:00<?, ?B/s]

ser_table.png:   0%|          | 0.00/326k [00:00<?, ?B/s]

tokens.json:   0%|          | 0.00/352k [00:00<?, ?B/s]

yue.mp3:   0%|          | 0.00/31.2k [00:00<?, ?B/s]

zh.mp3:   0%|          | 0.00/45.0k [00:00<?, ?B/s]

## Cell 5 — Fine-Tuning SenseVoice (Auto-Resume Checkpoint)

In [ ]:
import os, torch, glob, json

# Check for existing fine-tuned checkpoints
existing_ckpts = sorted(glob.glob(os.path.join(MODEL_OUTPUT, '*.pt*')))
ckpts = [c for c in existing_ckpts if 'model.pt' in c]
resume_ckpt = ckpts[-1] if ckpts else None
model_target = resume_ckpt if resume_ckpt else 'iic/SenseVoiceSmall'

if resume_ckpt:
    print(f'🔄 Fine-Tuning Checkpoint Found: Resuming from {os.path.basename(resume_ckpt)}')
else:
    print('🆕 Starting new Fine-Tuning session from iic/SenseVoiceSmall')

# Build Train & Validation dataset JSONL files with SenseVoice prefix
TRAIN_DATA_LIST = os.path.join(MODEL_OUTPUT, 'train_data.jsonl')
VAL_DATA_LIST = os.path.join(MODEL_OUTPUT, 'val_data.jsonl')

train_df = df_manifest[df_manifest['split'] == 'train'] if 'split' in df_manifest else df_manifest
val_df = df_manifest[df_manifest['split'] == 'val'] if 'split' in df_manifest else df_manifest.head(200)

with open(TRAIN_DATA_LIST, 'w', encoding='utf-8') as f:
    for _, row in train_df.iterrows():
        audio_p = os.path.join(NOISY_DIR, row['audio'])
        if os.path.exists(audio_p):
            f.write(json.dumps({'key': row['audio'], 'source': audio_p, 'target': f"<|zh|><|NEUTRAL|><|Speech|><|withitn|>{row['text']}"}, ensure_ascii=False) + '\n')

with open(VAL_DATA_LIST, 'w', encoding='utf-8') as f:
    for _, row in val_df.iterrows():
        audio_p = os.path.join(NOISY_DIR, row['audio'])
        if os.path.exists(audio_p):
            f.write(json.dumps({'key': row['audio'], 'source': audio_p, 'target': f"<|zh|><|NEUTRAL|><|Speech|><|withitn|>{row['text']}"}, ensure_ascii=False) + '\n')

print(f'✅ Fine-tuning datasets ready: Train ({len(train_df)}), Val ({len(val_df)}).')
print(f'💾 Checkpoint Output Directory: {MODEL_OUTPUT}')

# Execute FunASR Trainer with ++train_data_set_list AND ++valid_data_set_list
print('🚀 Starting fine-tuning loop on GPU... All checkpoints auto-flush to Google Drive.')
!python -m funasr.bin.train \
    ++model="{model_target}" \
    ++train_data_set_list="{TRAIN_DATA_LIST}" \
    ++valid_data_set_list="{VAL_DATA_LIST}" \
    ++output_dir="{MODEL_OUTPUT}" \
    ++train_conf.max_epoch=5 \
    ++train_conf.avg_nbest_model=0 \
    ++dataset_conf.batch_type="token" \
    ++dataset_conf.batch_size=2000 \
    ++optim_conf.lr=0.0001


## Cell 6 — Re-Evaluate Post Fine-Tuning

In [ ]:
import os, glob, torch, re, json, jiwer, pandas as pd
from funasr import AutoModel
from tqdm.notebook import tqdm

print('Evaluating Fine-Tuned Model...')
existing_ckpts = sorted(glob.glob(os.path.join(MODEL_OUTPUT, '*.pt*')))
ckpts = [c for c in existing_ckpts if 'model.pt' in c]

if not ckpts:
    print('ℹ Fine-tuned weights will be saved to Google Drive after running Cell 5.')
else:
    latest_ckpt = ckpts[-1]
    print(f'✅ Loading Fine-Tuned Weights from Local Project Directory: {os.path.basename(latest_ckpt)}')
    ft_model = AutoModel(
        model=MODEL_OUTPUT,
        init_param=latest_ckpt,
        vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
        vad_kwargs={'max_single_segment_time': 30000},
        device='cuda' if torch.cuda.is_available() else 'cpu',
        disable_update=True
    )
    print('✅ Fine-tuned project model loaded! Testing sample transcription...')

    test_audio = os.path.join(NOISY_DIR, df_manifest.iloc[0]['audio'])
    res = ft_model.generate(input=test_audio, language='zh', use_itn=True)
    print('Văn bản Gốc (GT)     :', df_manifest.iloc[0]['text'])
    print('Mô hình Dự Đoán (PRED):', res[0]['text'])
